<h1>Progetto Python di Danilo Manzo</h1>
<h2>Sistema di gestione prenotazioni — Orizon</h2>
Lo script consente di creare una base dati con attributi che descrivono un viaggio. Gli attributi di ogni elemento sono l'id utente, il paese di destinazione, la durata del viaggio in giorni, il costo di prenotazione, il numero di viaggiatori, la data di partenza e il tipo di viaggio.

Per gestire la base dati ho scelto di utilizzare una lista di dizionari. 

## Celle import

In [1]:
#servirà per gestire le date presenti nelle prenotazioni.
from datetime import datetime

## Definizione variabili principali
Lista per i viaggi e i tipi di viaggi ammessi.

In [2]:
# Creo una lista che conterrà un dizionario per ogni prenotazione.
gestione_viaggi = []

# Tipologie di viaggio accettate dal sistema (costante: non va modificata).
TIPI_VIAGGIO = ("nozze", "gruppo", "famiglia", "single")

## Funzione numero positivo

In [3]:
def numero_positivo(valore, tipi=(int,)):
    """
    Verifica che il valore sia un numero positivo dei tipi indicati.
    
    I booleani vengono esclusi anche se in Python sono sottoclassi di int.

    Parametri:
        valore: valore da controllare.
        tipi (tuple): tipi numerici ammessi. Di base solo int.

    Restituisce:
        bool: True se il valore è del tipo ammesso, non booleano e
        maggiore di zero, altrimenti False.
    """
    return isinstance(valore, tipi) and not isinstance(valore, bool) and valore > 0

## Registrazione di un nuovo viaggio
Funzione che consente di registrare un nuovo viaggio per un utente e di inserirlo in una lista di dizionari passata al momento della chiamata della funzione.

In [4]:
def prenota_viaggio(gestione_viaggi, id_utente, paese_destinazione, durata_g, costo_prenotazione, n_viaggiatori, data_partenza, tipo_viaggio):
    """
    Registra una nuova prenotazione nella lista indicata.

    Valida ogni parametro, normalizza paese e tipo di viaggio (minuscolo,
    senza spazi) e la data nel formato YYYY-MM-DD. Rifiuta le prenotazioni
    con stesso utente, paese e data di partenza di una già presente.
    In caso di inserimento di un parametro errato, stampa un messaggio
    e non modifica la lista.

    Parametri:
        gestione_viaggi (list): lista di dizionari in cui inserire la prenotazione.
        id_utente (int): identificativo dell'utente, maggiore di zero.
        paese_destinazione (str): paese di destinazione, testo non vuoto.
        durata_g (int): durata del viaggio in giorni, maggiore di zero.
        costo_prenotazione (int | float): costo della prenotazione, maggiore di zero.
        n_viaggiatori (int): numero di viaggiatori, maggiore di zero.
        data_partenza (str): data di partenza nel formato YYYY-MM-DD.
        tipo_viaggio (str): uno tra "nozze", "gruppo", "famiglia" e "single".

    Restituisce:
        bool: True se la prenotazione è stata registrata, altrimenti False.

    Nota:
        La cancellazione delle prenotazioni non è prevista.
    """

    #Validazione dei parametri passati.
    if not isinstance(gestione_viaggi, list):
        print("Il primo parametro deve essere una lista.")
        return False
    if not numero_positivo(id_utente):
        print("Id utente non valido: deve essere un numero intero maggiore di zero.")
        return False
    if not isinstance(paese_destinazione, str) or not paese_destinazione.strip():
        print("Paese di destinazione non valido: inserire un testo non vuoto.")
        return False
    if not numero_positivo(durata_g):
        print("Numero di giorni non valido: deve essere un numero intero maggiore di zero.")
        return False
    if not numero_positivo(costo_prenotazione, (int, float)):
        print("Costo non valido: deve essere un numero maggiore di zero.")
        return False
    if not numero_positivo(n_viaggiatori):
        print("Numero di viaggiatori non valido: deve essere un numero intero maggiore di zero.")
        return False
    if not isinstance(data_partenza, str):
        print("Data non valida: deve essere un testo nel formato YYYY-MM-DD.")
        return False
    # strptime solleva ValueError se la data non è valida
    try:
        data = datetime.strptime(data_partenza, "%Y-%m-%d").strftime("%Y-%m-%d") #la data viene prima validata e poi ritrasformata in testo.
    except ValueError:
        print("Data non valida: usare il formato YYYY-MM-DD con una data esistente.")
        return False
    if not isinstance(tipo_viaggio, str) or tipo_viaggio.strip().lower() not in TIPI_VIAGGIO: #non verifico la coerenza tra tipo di viaggio e numero di viaggiatori
        print("Tipo di viaggio non identificato. Inserirne uno tra \"nozze\", \"gruppo\", \"famiglia\", \"single\".")
        return False
    
    #normalizzazione dei parametri paese_destinazione
    paese = paese_destinazione.strip().lower() # Il paese è salvato in minuscolo e senza spazi per facilitare il controllo duplicati.
    
    #Controllo duplicati all'interno della lista dei viaggi prenotati.
    for prenotazione in gestione_viaggi:
        if (prenotazione["id_utente"] == id_utente
                and prenotazione["paese_destinazione"] == paese
                and prenotazione["data_partenza"] == data):
            print(f"Prenotazione rifiutata: l'utente {id_utente} ha già un viaggio "
                  f"per {paese.capitalize()} con data {data}")
            return False

    #inserimento della nuova prenotazione nella lista.
    gestione_viaggi.append({"id_utente": id_utente,
                            "id_viaggio": len(gestione_viaggi) + 1, # L'id viene generato in modo progressivo; non è prevista la cancellazione. Gli ID non si duplicano finché non vengono cancellate le prenotazioni
                            "paese_destinazione": paese,
                            "durata_g": durata_g,
                            "costo_prenotazione": costo_prenotazione,
                            "n_viaggiatori": n_viaggiatori,
                            "data_partenza": data,
                            "tipo_viaggio": tipo_viaggio.lower().strip()
    })
    return True

## Prove di prenotazioni effettuate su una lista_test = []

In [5]:
lista_test = []

# Prima prenotazione: deve essere accettata
if prenota_viaggio(lista_test, 1, "Argentina", 15, 1500, 2, "2026-05-05", "nozze"):
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

# Stessa prenotazione: duplicato
if prenota_viaggio(lista_test, 1, "Argentina", 15, 1500, 2, "2026-05-05", "nozze"): #Esito atteso: 1° prenotazione rifiutata
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

# Paese in maiuscolo: duplicato
if prenota_viaggio(lista_test, 1, "ARGENTINA", 15, 1500, 2, "2026-05-05", "nozze"): #Esito atteso: 2° prenotazione rifiutata
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#Paese contenente spazi: duplicato
if prenota_viaggio(lista_test, 1, " argentina ", 15, 1500, 2, "2026-05-05", "nozze"): #Esito atteso: 3° prenotazione rifiutata
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

# Data con una sola cifra per mesi e giorni: duplicato
if prenota_viaggio(lista_test, 1, "Argentina", 15, 1500, 2, "2026-5-5", "nozze"): #Esito atteso: 4° prenotazione rifiutata
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

# Data non valida
if prenota_viaggio(lista_test, 1, "Argentina", 15, 1500, 2, "2026-2-30", "nozze"): #Esito atteso: 5° prenotazione rifiutata
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

# Un solo campo diverso: deve essere accettata. Esito atteso: prenotazioni accettate
if prenota_viaggio(lista_test, 2, "Argentina", 15, 1500, 2, "2026-05-05", "nozze"): #solo utente_id diverso 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")
if prenota_viaggio(lista_test, 2, "Cile", 15, 1500, 2, "2026-05-05", "nozze"): #solo Paese diverso
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")
if prenota_viaggio(lista_test, 2, "Argentina", 15, 1500, 2, "2026-12-07", "nozze"): #solo data diversa
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

print("Lunghezza della lista test uguale a", len(lista_test))   #Esito atteso: 4 prenotazioni accettate

1 ha prenotato con successo

Prenotazione rifiutata: l'utente 1 ha già un viaggio per Argentina con data 2026-05-05
Prenotazione rifiutata: l'utente 1 ha già un viaggio per Argentina con data 2026-05-05
Prenotazione rifiutata: l'utente 1 ha già un viaggio per Argentina con data 2026-05-05
Prenotazione rifiutata: l'utente 1 ha già un viaggio per Argentina con data 2026-05-05
Data non valida: usare il formato YYYY-MM-DD con una data esistente.
2 ha prenotato con successo

2 ha prenotato con successo

2 ha prenotato con successo

Lunghezza della lista test uguale a 4


## Controllo validazione dati su durata, paese, data di partenza e tipo di viaggio inesistente
Durata del viaggio = str  
Paese di destinazione = None  
Paese di destinazione = ""  
data di partenza = str  
durata del viaggio = True  
tipo di viaggio inesistente

In [6]:
lista_test = []

#durata in giorni espressa a parole. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "Cina", "dieci", 2000, 3, "2026-02-17", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#Valore del Paese None. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, None, 10, 2000, 3, "2026-02-17", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#Stringa vuota per il parametro Paese di destinazione. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "", 10, 2000, 3, "2026-02-17", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#Valore stringa per il parametro data di partenza. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "Cina", 10, 2000, 3, "Boh", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#Valore True per il parametro durata in giorni. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "Cina", True, 2000, 3, "2026-12-03", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#tipo di viaggio non presente nella lista TIPI_VIAGGIO. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "Cina", 10, 2000, 3, "2026-12-03", "svago"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#Valore stringa per il numero di viaggiatori. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "Cina", 10, 2000, "tre", "2026-12-03", "noz"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

Numero di giorni non valido: deve essere un numero intero maggiore di zero.
Paese di destinazione non valido: inserire un testo non vuoto.
Paese di destinazione non valido: inserire un testo non vuoto.
Data non valida: usare il formato YYYY-MM-DD con una data esistente.
Numero di giorni non valido: deve essere un numero intero maggiore di zero.
Tipo di viaggio non identificato. Inserirne uno tra "nozze", "gruppo", "famiglia", "single".
Numero di viaggiatori non valido: deve essere un numero intero maggiore di zero.


## Ultime validazioni
Costo uguale a 0 o negativo.  
Durata di tipo float.  
Tipo di viaggio passato come valore None.  
Id utente uguale a 0.

In [7]:
lista_test = []

#Costo prenotazione 0 o negativo. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "Cina", 10, -2000, 3, "2026-12-03", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")
if prenota_viaggio(lista_test, 3, "Cina", 10, 0, 3, "2026-12-03", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#durata del viaggio espressa in numeri decimali. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "Cina", 10.5, 2000, 3, "2026-12-03", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#tipo di viaggio con valore None. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 3, "Cina", 10, 2000, 3, "2026-12-03", None): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

#Id_utente con valore 0. Esito atteso: prenotazione rifiutata
if prenota_viaggio(lista_test, 0, "Cina", 10, 2000, 3, "2026-12-03", "famiglia"): 
    print(lista_test[-1]['id_utente'], "ha prenotato con successo\n")

Costo non valido: deve essere un numero maggiore di zero.
Costo non valido: deve essere un numero maggiore di zero.
Numero di giorni non valido: deve essere un numero intero maggiore di zero.
Tipo di viaggio non identificato. Inserirne uno tra "nozze", "gruppo", "famiglia", "single".
Id utente non valido: deve essere un numero intero maggiore di zero.


## Stampa una singola prenotazione

In [8]:
def stampa_prenotazione(prenotazione):
    """
    Stampa una singola prenotazione in un formato facilmente leggibile.

    Parametri:
        prenotazione (dict):dizionario con i dati di una prenotazione.
    """
    
    print(f"ID utente: {prenotazione['id_utente']}\nPaese di destinazione: {prenotazione['paese_destinazione'].title()}\n"
                  f"Durata del viaggio in giorni: {prenotazione['durata_g']}\nCosto prenotazione: {prenotazione['costo_prenotazione']}\n"
                  f"Numero viaggiatori: {prenotazione['n_viaggiatori']}\nData di partenza: {prenotazione['data_partenza']}\n"
                  f"Tipo di viaggio: {prenotazione['tipo_viaggio'].title()}\nID del viaggio: {prenotazione['id_viaggio']}\n______")

## Visualizzazione delle prenotazioni
Visualizzerà tutte le prenotazioni di una lista passata come parametro.

In [9]:
def visualizza_prenotazioni(gestione_viaggi):
    """
    Stampa tutte le prenotazioni della lista, una alla volta.

    Se la lista è vuota, stampa un messaggio al posto delle prenotazioni.

    Parametri:
        gestione_viaggi (list): lista di dizionari da mostrare.
    """
    
    if gestione_viaggi:
        for prenotazione in gestione_viaggi:
            stampa_prenotazione(prenotazione)
    else:
        print("Nessuna prenotazione da mostrare.\n")

## Funzione di ricerca elementi e filtraggio avanzato
Cerca le prenotazioni all'interno di una lista specificata dall'utente e secondo dei parametri personalizzabili.
L'inserimento dei parametri è facoltativo.

In [22]:
def cerca_viaggi(gestione_viaggi, paese_destinazione=None, tipo_viaggio=None, costo_massimo=None, durata_minima=None):
    """
    Cerca le prenotazioni che soddisfano tutti i criteri indicati.

    I criteri sono facoltativi: quelli lasciati a None non vengono
    applicati. Paese supporta la ricerca parziale
    e non distingue tra maiuscole e minuscole.
    Tipo di viaggio è un testo da cercare in nella tupla
    TIPI_VIAGGIO.

    Parametri:
        gestione_viaggi (list): lista di dizionari in cui cercare.
        paese_destinazione (str): testo da cercare nel paese di destinazione.
        tipo_viaggio (str): testo da cercare nel tipo di viaggio.
        costo_massimo (int | float): costo massimo della prenotazione.
        durata_minima (int): durata minima del viaggio, in giorni.

    Restituisce:
        list: le prenotazioni che soddisfano i criteri; se nessuna
        prenotazione corrisponde o se un criterio non è valido rimanda indietro
        una lista vuota.
    """
    #validazione parametri passati
    if paese_destinazione is not None and (not isinstance(paese_destinazione, str) or not paese_destinazione.strip()):
        print("Paese di destinazione non valido: inserire un testo non vuoto.")
        return []
    if tipo_viaggio is not None and (not isinstance(tipo_viaggio, str) or tipo_viaggio.strip().lower() not in TIPI_VIAGGIO):
        print("Tipo di viaggio non identificato. Inserirne uno tra \"nozze\", \"gruppo\", \"famiglia\", \"single\".")
        return []
    if costo_massimo is not None and not numero_positivo(costo_massimo,(int, float)):
        print("Costo massimo non valido: deve essere un numero maggiore di zero.")
        return []
    if durata_minima is not None and not numero_positivo(durata_minima):
        print("Durata minima non valida: deve essere un numero intero maggiore di zero.")
        return []

    # Normalizzazione dei criteri testuali: minuscolo e senza spazi per il confronto con i dati salvati.
    paese = paese_destinazione.strip().lower() if paese_destinazione is not None else None
    tipo = tipo_viaggio.strip().lower() if tipo_viaggio is not None else None

    #ciclo di controllo for per confrontare i parametri passati alla funzione con i valori presenti nella lista passata alla funzione.
    viaggi_filtrati = []
    for prenotazione in gestione_viaggi:
        if paese is not None and paese not in prenotazione["paese_destinazione"]:
            continue
        if tipo is not None and tipo != prenotazione["tipo_viaggio"]:
            continue
        if costo_massimo is not None and prenotazione["costo_prenotazione"] > costo_massimo:
            continue
        if durata_minima is not None and prenotazione["durata_g"] < durata_minima:
            continue
        viaggi_filtrati.append(prenotazione)
                
    return viaggi_filtrati

## Funzione statistica
Calcola il totale delle prenotazioni, il numero di viaggiatori per paese, la distribuzione per tipo di viaggio, le destinazioni più richieste, il numero massimo di prenotazioni per un paese e l'ultima prenotazione registrata.

In [11]:
def statistiche(gestione_viaggi):
    """
    Calcola le statistiche sulle prenotazioni della lista.

    Parametri:
        gestione_viaggi (list): lista di dizionari, uno per prenotazione.

    Restituisce:
        dict | None: dizionario con totale prenotazioni, viaggiatori per
        paese, distribuzione per tipo di viaggio, paesi più frequenti 
        (inclusi i pari merito) e ultima prenotazione registrata; None se la lista è vuota.
    """
    
    if not gestione_viaggi:
        print("Nessuna prenotazione presente: impossibile calcolare le statistiche.")
        return None

    #inizializzazioni dei dizionari che mi conterranno le statistiche da restituire
    viaggiatori_per_paese = {}
    prenotazioni_per_paese = {}
    distribuzione_tipo_viaggio = {}

    #Ciclo for in cui scorro la lista delle prenotazioni e aggiungo le chiavi e i valori all'interno dei rispettivi dizionari.
    for prenotazione in gestione_viaggi:
        paese = prenotazione["paese_destinazione"]
        tipo = prenotazione["tipo_viaggio"]
        viaggiatori_per_paese[paese] = viaggiatori_per_paese.get(paese, 0) + prenotazione["n_viaggiatori"]
        prenotazioni_per_paese[paese] = prenotazioni_per_paese.get(paese, 0) + 1
        distribuzione_tipo_viaggio[tipo] = distribuzione_tipo_viaggio.get(tipo, 0) + 1

    #I Paesi con il massimo numero di prenotazioni (inclusi tutti i pari merito)
    massimo = max(prenotazioni_per_paese.values())
    paesi_piu_frequenti = [paese for paese, n in prenotazioni_per_paese.items() if n == massimo]

    return {
        "totale_prenotazioni": len(gestione_viaggi),
        "viaggiatori_per_paese": viaggiatori_per_paese,
        "distribuzione_tipo_viaggio": distribuzione_tipo_viaggio,
        "paesi_piu_frequenti": paesi_piu_frequenti,
        "prenotazioni_per_paese_max": massimo,
        "ultima_registrata": gestione_viaggi[-1] # ultima prenotazione inserita nella lista
    }

## Stampa il risultato della funzione statistiche()

In [12]:
def stampa_statistiche(risultato):
    """
    Stampa in formato leggibile le statistiche calcolate da statistiche().

    Se risultato è None (lista delle prenotazioni vuota) non stampa nulla:
    l'avviso è già stato mostrato da statistiche().

    Parametri:
        risultato (dict | None): valore restituito da statistiche().
    """
    
    if risultato is None:
        return
    print("=== Statistiche prenotazioni ===\n")

    print(f"Prenotazioni totali: {risultato['totale_prenotazioni']}\n")

    print("Viaggiatori per paese:")
    viaggiatori = risultato["viaggiatori_per_paese"]
    for paese in sorted(viaggiatori, key=viaggiatori.get, reverse=True):
        print(f"  {paese.title()}: {viaggiatori[paese]}")

    paesi = ", ".join(p.title() for p in risultato["paesi_piu_frequenti"])
    print(f"\nDestinazioni più richieste: {paesi} ({risultato['prenotazioni_per_paese_max']} prenotazioni)")

    print("\nPrenotazioni per tipo di viaggio:")
    distribuzione = risultato["distribuzione_tipo_viaggio"]
    for tipo in sorted(distribuzione, key=distribuzione.get, reverse=True):
        print(f"  {tipo.title()}: {distribuzione[tipo]}")

    print("\nUltima prenotazione registrata:")
    stampa_prenotazione(risultato["ultima_registrata"])

## Dati di esempio
10 prenotazioni inserite nella lista gestione_viaggi

In [13]:
# Dati di esempio: una tupla per prenotazione, nello stesso ordine
# dei parametri di prenota_viaggio (esclusa la lista).
dati_esempio = [
    (1, "Argentina", 15, 1500, 2, "2026-11-20", "nozze"),
    (2, "Argentina", 10, 1100, 5, "2026-11-25", "famiglia"),
    (3, "Argentina", 9, 1000, 7, "2026-11-30", "gruppo"),
    (4, "Cile", 20, 2000, 6, "2026-12-15", "gruppo"),
    (5, "Cile", 17, 1800, 4, "2026-12-07", "famiglia"),
    (6, "Cile", 14, 1400, 2, "2026-12-03", "nozze"),
    (1, "Germania", 7, 800, 3, "2026-12-20", "famiglia"),
    (7, "Germania", 9, 500, 1, "2026-12-17", "single"),
    (8, "Italia", 10, 1000, 5, "2026-12-14", "gruppo"),
    (2, "Spagna", 11, 1100, 1, "2026-12-18", "single"),
]

registrate = 0
for dati in dati_esempio:
    if prenota_viaggio(gestione_viaggi, *dati):
        registrate += 1

print(f"{registrate} prenotazioni registrate su {len(dati_esempio)}.")

10 prenotazioni registrate su 10.


## Validazione funzione cerca_viaggi

In [14]:
visualizza_prenotazioni(cerca_viaggi(gestione_viaggi))

ID utente: 1
Paese di destinazione: Argentina
Durata del viaggio in giorni: 15
Costo prenotazione: 1500
Numero viaggiatori: 2
Data di partenza: 2026-11-20
Tipo di viaggio: Nozze
ID del viaggio: 1
______
ID utente: 2
Paese di destinazione: Argentina
Durata del viaggio in giorni: 10
Costo prenotazione: 1100
Numero viaggiatori: 5
Data di partenza: 2026-11-25
Tipo di viaggio: Famiglia
ID del viaggio: 2
______
ID utente: 3
Paese di destinazione: Argentina
Durata del viaggio in giorni: 9
Costo prenotazione: 1000
Numero viaggiatori: 7
Data di partenza: 2026-11-30
Tipo di viaggio: Gruppo
ID del viaggio: 3
______
ID utente: 4
Paese di destinazione: Cile
Durata del viaggio in giorni: 20
Costo prenotazione: 2000
Numero viaggiatori: 6
Data di partenza: 2026-12-15
Tipo di viaggio: Gruppo
ID del viaggio: 4
______
ID utente: 5
Paese di destinazione: Cile
Durata del viaggio in giorni: 17
Costo prenotazione: 1800
Numero viaggiatori: 4
Data di partenza: 2026-12-07
Tipo di viaggio: Famiglia
ID del viaggi

Validazione cerca_viaggi() con paese_destinazione intero e stringa vuota.  
Costo massimo uguale ad un numero negativo.

In [15]:
#cerca_viaggi(gestione_viaggi, paese_destinazione=None, tipo_viaggio=None, costo_massimo=None, durata_minima=None)
visualizza_prenotazioni(cerca_viaggi(gestione_viaggi, 123))
visualizza_prenotazioni(cerca_viaggi(gestione_viaggi, ""))
visualizza_prenotazioni(cerca_viaggi(gestione_viaggi, paese_destinazione="argen", costo_massimo=-5))

Paese di destinazione non valido: inserire un testo non vuoto.
Nessuna prenotazione da mostrare.

Paese di destinazione non valido: inserire un testo non vuoto.
Nessuna prenotazione da mostrare.

Costo massimo non valido: deve essere un numero maggiore di zero.
Nessuna prenotazione da mostrare.



Ricerca parziale per paese di destinazione tramite la stringa 'argen'

In [16]:
visualizza_prenotazioni(cerca_viaggi(gestione_viaggi,"argen"))

ID utente: 1
Paese di destinazione: Argentina
Durata del viaggio in giorni: 15
Costo prenotazione: 1500
Numero viaggiatori: 2
Data di partenza: 2026-11-20
Tipo di viaggio: Nozze
ID del viaggio: 1
______
ID utente: 2
Paese di destinazione: Argentina
Durata del viaggio in giorni: 10
Costo prenotazione: 1100
Numero viaggiatori: 5
Data di partenza: 2026-11-25
Tipo di viaggio: Famiglia
ID del viaggio: 2
______
ID utente: 3
Paese di destinazione: Argentina
Durata del viaggio in giorni: 9
Costo prenotazione: 1000
Numero viaggiatori: 7
Data di partenza: 2026-11-30
Tipo di viaggio: Gruppo
ID del viaggio: 3
______


Filtro avanzato: Cile, tipo famiglia, costo max 2000, almeno 15 giorni:

In [17]:
visualizza_prenotazioni(cerca_viaggi(gestione_viaggi, paese_destinazione="cile",
                                     tipo_viaggio="famiglia", costo_massimo=2000,
                                     durata_minima=15))

ID utente: 5
Paese di destinazione: Cile
Durata del viaggio in giorni: 17
Costo prenotazione: 1800
Numero viaggiatori: 4
Data di partenza: 2026-12-07
Tipo di viaggio: Famiglia
ID del viaggio: 5
______


Ricerca tramite una lista vuota:

In [18]:
lista_vuota = []
visualizza_prenotazioni(cerca_viaggi(lista_vuota))

Nessuna prenotazione da mostrare.



Ricerca valida senza risultati (Giappone):

In [19]:
visualizza_prenotazioni(cerca_viaggi(gestione_viaggi, paese_destinazione="giappone"))

Nessuna prenotazione da mostrare.



## Verifica della funzione statistica

In [20]:
stampa_statistiche(statistiche(gestione_viaggi))

=== Statistiche prenotazioni ===

Prenotazioni totali: 10

Viaggiatori per paese:
  Argentina: 14
  Cile: 12
  Italia: 5
  Germania: 4
  Spagna: 1

Destinazioni più richieste: Argentina, Cile (3 prenotazioni)

Prenotazioni per tipo di viaggio:
  Famiglia: 3
  Gruppo: 3
  Nozze: 2
  Single: 2

Ultima prenotazione registrata:
ID utente: 2
Paese di destinazione: Spagna
Durata del viaggio in giorni: 11
Costo prenotazione: 1100
Numero viaggiatori: 1
Data di partenza: 2026-12-18
Tipo di viaggio: Single
ID del viaggio: 10
______


Funzione statistiche applicata su una lista vuota

In [21]:
lista_vuota = []
stampa_statistiche(statistiche(lista_vuota))

Nessuna prenotazione presente: impossibile calcolare le statistiche.
